# 02 — Data Cleaning

Applies cleaning rules to the raw event file, logs all transformations, and saves a cleaned Parquet.

In [ ]:
import sys, os
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings('ignore')
import pandas as pd


## 1 · Load Raw Data

In [ ]:
df_raw = pd.read_parquet('../data/transit_events.parquet')
print(f"Raw shape : {df_raw.shape}")
print("\nNull counts (raw):")
print(df_raw.isnull().sum()[df_raw.isnull().sum() > 0])


## 2 · Run Cleaning Pipeline

In [ ]:
import cleaning

df_clean, log = cleaning.clean(df_raw, log=True)
print(f"Clean shape: {df_clean.shape}")


## 3 · Cleaning Log

In [ ]:
print("=== Cleaning Log ===")
for entry in log:
    print(" •", entry)


## 4 · Before / After Row Counts

In [ ]:
dropped = len(df_raw) - len(df_clean)
print(f"Rows before : {len(df_raw):>10,}")
print(f"Rows after  : {len(df_clean):>10,}")
print(f"Rows dropped: {dropped:>10,}  ({dropped / len(df_raw) * 100:.3f} %)")


## 5 · Null Counts — Before vs After

In [ ]:
null_before = df_raw.isnull().sum()
null_after  = df_clean.isnull().sum()

comparison = pd.DataFrame({
    'before': null_before,
    'after' : null_after,
}).query('before > 0 or after > 0')

print(comparison if not comparison.empty else "No nulls before or after.")


## 6 · Save Cleaned Parquet

In [ ]:
out_path = '../data/transit_events_clean.parquet'
df_clean.to_parquet(out_path, index=False)
print(f"Saved → {out_path}")


## 7 · Final Verification

In [ ]:
critical_cols = [
    'timestamp', 'station_id', 'route_id',
    'delay_minutes', 'next_hour_avg_delay',
]
for col in critical_cols:
    n_null = df_clean[col].isnull().sum()
    assert n_null == 0, f"Column '{col}' still has {n_null} nulls after cleaning."
    print(f"  {col}: 0 nulls ✓")

print("\nAll critical columns are null-free.")
